# Introduction

dpstats is a Python library for implementing DP algorithms for common statistical tasks and interfacing with the dpstatsbench benchmark. It includes built-in algorithms for common statistical problems and lets you implement your own using the same interface.

In this notebook, we introduce how the library represents sensitive data, public information, and privacy budgets, then show how to run and evaluate a differentially private algorithm.

## Tasks

[Tasks](https://saminulh.github.io/dpstats/library/reference/tasks/dpstats.task.Task/) are the central organizing concept of this library. Each task corresponds to a common statistical problem, such as [mean estimation](https://saminulh.github.io/dpstats/mean_estimation/) or [linear regression](https://saminulh.github.io/dpstats/linear_regression/).

A task specifies what problem we want to solve, while an algorithm specifies how to solve it. Each [Task](https://saminulh.github.io/dpstats/library/reference/tasks/dpstats.task.Task/) defines the types of the sensitive input, the available public information (metadata), and the output an algorithm returns. Algorithms for the same task share these types, allowing us to compare different methods on the same problem.

An [Instance](https://saminulh.github.io/dpstats/library/reference/instance/dpstats.instance.Instance/) represents one concrete problem for a task, identifying its input and metadata. Below, we work directly with these two objects to run an algorithm for the mean estimation task.

## Example: Mean Estimation

The mean estimation task is simply to estimate the mean of a set of d-dimensional samples.
This has a straightforward type for the input and output: the input is a 2d array of samples, and the output is expected to be a 1d array representing the mean estimate.

We now turn to the available public information, or metadata.
In this library, we treat the number of samples as known (see our [privacy interpretation](https://saminulh.github.io/dpstats/library/reference/algorithm/dpstats.algorithm.PrivacyInterpretation/) page for details).
We also often have publicly known constraints on the space of allowed values for each sample, such as a one-hot block of coordinates or the coordinates lying inside a hypercube. We represent this using a [Domain](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.Domain/) object (see the [Domain notebook](https://saminulh.github.io/dpstats/library/tutorials/domains/) for more details).

Summarizing the above, for the [MeanEstimation](https://saminulh.github.io/dpstats/mean_estimation/) task, we have the following types:
```python
class MeanEstimationInput(TaskInput):
    X: Array2D

class MeanEstimationMetadata(TaskMetadata):
    n_samples: int
    domain: Domain

class MeanEstimationOutput(TaskOutput):
    mean_est: Array1D
```

For other tasks, you can visit their [Definitions] page to see the definitions of the input, metadata, and output types.

### Constructing input and metadata

In the following example, the metadata expects 3 one-dimensional samples lying in [0, 1] (using [BoxDomain](https://saminulh.github.io/dpstats/library/reference/domains/dpstats.domains.BoxDomain/)).
The first input we construct satisfies this, while the other two do not.
We use the `validate_input` method of the metadata object to check if the input is valid for the given metadata.

In [ ]:
import numpy as np

from dpstats.domains import BoxDomain
from dpstats.tasks.mean_estimation import (
    MeanEstimationInput,
    MeanEstimationMetadata,
    MeanEstimationOutput,
)

metadata = MeanEstimationMetadata(
    n_samples=3,
    domain=BoxDomain(bounds=np.array([[0.0, 1.0]])),  # the domain is [0, 1]
)
valid_input = MeanEstimationInput(X=np.array([[0.2], [0.5], [0.8]]))
four_samples = MeanEstimationInput(X=np.array([[0.2], [0.5], [0.8], [0.9]]))
out_of_bounds = MeanEstimationInput(X=np.array([[0.2], [0.5], [1.2]]))

for label, candidate in (
    ("first input", valid_input),
    ("second input", four_samples),
    ("third input", out_of_bounds),
):
    try:
        metadata.validate_input(candidate)
    except ValueError as error:
        print(f"{label}: {error}")
    else:
        print(f"{label}: valid")


### Running an algorithm

All [MeanEstimation](https://saminulh.github.io/dpstats/mean_estimation/) algorithms implement a `run` function with the following signature:
```python
def run(
    self,
    metadata: MeanEstimationMetadata,
    input: MeanEstimationInput,
    budget: DPBudget,
    seed: int,
) -> MeanEstimationOutput:
    ...
```

Note that it takes in the metadata, input, and privacy budget as arguments. It also takes in a seed for randomness.


We demonstrate this below, where we run the built-in [GaussianMechanism](https://saminulh.github.io/dpstats/mean_estimation/algorithms/GaussianMechanism/) algorithm with the previous input and metadata. For more built-in algorithms as well as implementing your own, see the [Algorithms notebook](https://saminulh.github.io/dpstats/library/tutorials/algorithms/).

We use an [ApproxDPBudget](https://saminulh.github.io/dpstats/library/reference/budgets/dpstats.budgets.ApproxDPBudget/) with $\varepsilon = 1$ and $\delta = 10^{-6}$. We discuss privacy budgets in more detail in the [Privacy budgets and accounting notebook](https://saminulh.github.io/dpstats/library/tutorials/privacy_budgets_and_accounting/).

In [ ]:
from dpstats.budgets import ApproxDPBudget
from dpstats.tasks.mean_estimation.algorithms import GaussianMechanism

algorithm = GaussianMechanism()
budget = ApproxDPBudget(eps=1.0, delta=1e-6)
output1 = algorithm.run(metadata=metadata, input=valid_input, budget=budget, seed=42)
output2 = algorithm.run(metadata=metadata, input=valid_input, budget=budget, seed=43)

print(
    "output is a MeanEstimationOutput object:",
    isinstance(output1, MeanEstimationOutput),
)
print("output1 mean_est:", output1.mean_est)
print("output2 mean_est:", output2.mean_est)
print("sample mean:", np.mean(valid_input.X, axis=0))


### Measuring the output

To evaluate an algorithm’s output, we use a [Metric](https://saminulh.github.io/dpstats/library/reference/metric/dpstats.metric.Metric/). For mean estimation, one choice is the distance between the estimated mean and the sample mean, which we access via
```python
from dpstats.tasks import MeanEstimation, list_built_in_metrics

metric = list_built_in_metrics(MeanEstimation)["l2_error_to_sample_mean"]
```
This metric require passing the sample mean as an additional `sample_mean` argument.

The next cell measures this error for each of our two outputs without rerunning the algorithm.

The [Metrics and baselines notebook](https://saminulh.github.io/dpstats/library/tutorials/metrics_and_baselines/) introduces other metrics, the metric argument values they require, and how to compare errors against a baseline.

In [ ]:
from dpstats.tasks import MeanEstimation, list_built_in_metrics

metric = list_built_in_metrics(MeanEstimation)["l2_error_to_sample_mean"]
sample_mean = np.mean(valid_input.X, axis=0)

print("output1 error:", metric.calculate(output1, sample_mean=sample_mean))
print("output2 error:", metric.calculate(output2, sample_mean=sample_mean))


## Next steps
This completes the overview of the most basic concepts in this library.
Next, the [Algorithms notebook](https://saminulh.github.io/dpstats/library/tutorials/algorithms/) goes through running and creating algorithms in more detail.